In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import joblib

DB = "resqfood.db"
OUT_CSV = "resqfood_processed.csv"
FEATURE_FILE = "feature_columns.pkl"

print("Loading data from database...")

try:
    with sqlite3.connect(DB) as conn:
        df = pd.read_sql("SELECT * FROM listings", conn)
except Exception as e:
    raise RuntimeError(f"Database load failed: {e}")

print(f"Loaded {len(df)} rows")

if df.empty:
    raise ValueError("Dataset is empty!")

if "listing_id" in df.columns:
    df = df.drop_duplicates(subset=["listing_id"]).reset_index(drop=True)

if "time_posted" in df.columns:
    df["time_posted"] = pd.to_datetime(df["time_posted"], errors="coerce")

numeric_cols = ["distance_km", "ambient_temp_c", "quantity"]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val if not np.isnan(median_val) else 0)

for col in ["distance_km", "quantity"]:
    if col not in df.columns:
        df[col] = 0

categorical_cols = [
    "donor_type", "food_type", "unit",
    "city", "packaging", "donor_reliability"
]

for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).fillna("Unknown")

print("Creating features...")

# Time features
if "time_posted" in df.columns:
    df["hour_posted"] = df["time_posted"].dt.hour.fillna(0)
else:
    df["hour_posted"] = 0

df["is_peak_hour"] = ((df["hour_posted"] >= 8) & (df["hour_posted"] <= 20)).astype(int)

# Quantity conversion
unit_to_kg = {"kg": 1.0, "plates": 0.5, "boxes": 2.0, "packs": 1.0}

if "unit" in df.columns:
    df["quantity_kg"] = df["quantity"] * df["unit"].map(unit_to_kg).fillna(1.0)
else:
    df["quantity_kg"] = df["quantity"]

# Reliability score
rel_map = {"Low": 0.5, "Medium": 0.8, "High": 1.0}

if "donor_reliability" in df.columns:
    df["donor_reliability_score"] = df["donor_reliability"].map(rel_map).fillna(0.7)
else:
    df["donor_reliability_score"] = 0.7

# Time since cooked
if "time_since_cooked_mins" in df.columns:
    df["time_since_cooked_hours"] = pd.to_numeric(
        df["time_since_cooked_mins"], errors="coerce"
    ).fillna(60) / 60
else:
    df["time_since_cooked_hours"] = 1

# Interaction feature (log safe)
df["qty_div_distance"] = np.log1p(
    df["quantity_kg"] / (df["distance_km"] + 0.5)
)

print("Encoding...")

encode_cols = ["donor_type", "food_type", "city", "packaging"]
encode_cols = [c for c in encode_cols if c in df.columns]

df_model = pd.get_dummies(df, columns=encode_cols, drop_first=True)

print("Selecting features...")

base_features = [
    "quantity_kg",
    "distance_km",
    "ambient_temp_c",
    "time_since_cooked_hours",
    "donor_reliability_score",
    "is_peak_hour",
    "qty_div_distance"
]

base_features = [f for f in base_features if f in df_model.columns]

encoded_features = [
    c for c in df_model.columns
    if c.startswith(("donor_type_", "food_type_", "city_", "packaging_"))
]

features = base_features + encoded_features

if not features:
    raise ValueError("No features available!")

df_model = df_model.reindex(columns=features, fill_value=0)

X = df_model

joblib.dump(features, FEATURE_FILE)
print(f"Saved feature schema → {FEATURE_FILE}")

processed = X.copy()

if "picked_up" in df.columns:
    processed["picked_up"] = df["picked_up"]

if "freshness_score" in df.columns:
    processed["freshness_score"] = df["freshness_score"]

# Metadata
if "listing_id" in df.columns:
    processed["listing_id"] = df["listing_id"]

if "time_posted" in df.columns:
    processed["time_posted"] = df["time_posted"]

print(f"Final dataset shape: {processed.shape}")
print(f"Total features used: {len(features)}")


processed.to_csv(OUT_CSV, index=False)
print(f"Saved CSV → {OUT_CSV}")

with sqlite3.connect(DB) as conn:
    processed.to_sql("listings_clean", conn, if_exists="replace", index=False)

print("Saved table → listings_clean")
print("Pipeline completed successfully.")